# Chapter 1 — Pythonic Thinking

Source: *Effective Python*, 3rd Edition (Brett Slatkin), Chapter 1.

This chapter covers the small habits that separate "code that works" from "code a Python programmer would recognize as Python" — version awareness, style, and a handful of expressions that read cleanly instead of cleverly.

---


### Item 1 — Know Which Version of Python You're Using

Python 2 reached end-of-life in 2020; all real work today targets Python 3, but *which* Python 3 matters — syntax like the walrus operator (`:=`, 3.8+) or `match` statements (3.10+) simply doesn't exist on older interpreters. Check the version with `sys.version` (a human-readable string) or `sys.version_info` (a comparable tuple you can use in `if` checks to guard version-specific code).


In [1]:
import sys

print(sys.version)
print(sys.version_info)

# version_info supports tuple comparison -- useful for guarding version-specific code
if sys.version_info >= (3, 10):
    print('match statements are available (3.10+)')
if sys.version_info >= (3, 8):
    print('the walrus operator := is available (3.8+)')


3.10.12 (main, Jun 22 2026, 18:55:27) [GCC 11.4.0]
sys.version_info(major=3, minor=10, micro=12, releaselevel='final', serial=0)
match statements are available (3.10+)
the walrus operator := is available (3.8+)


### Item 2 — Follow the PEP 8 Style Guide

PEP 8 is Python's official style guide, and following it matters because most Python code you'll ever read (libraries, coworkers' code, AI-generated code) assumes it. Highlights: 4 spaces per indent level (never tabs), lines under 79-99 characters, `lowercase_with_underscores` for functions/variables, `CapWords` for classes, `_leading_underscore` for internal-use names, blank lines to separate logical sections, spaces around operators (`x = 1` not `x=1`) but not inside brackets touching the item. Tools like `pycodestyle`/`flake8`/`ruff` check this automatically instead of relying on memory.


In [2]:
# PEP 8 examples: left is compliant, right (as a string, not run) shows what to avoid
def calculate_total(item_price, tax_rate):   # lowercase_with_underscores, spaces around =
    return item_price * (1 + tax_rate)

class ShoppingCart:                           # CapWords for classes
    def __init__(self):
        self._items = []                      # leading underscore: internal use

print(calculate_total(10, 0.08))

non_compliant_example = '''
def calculateTotal(itemPrice,taxRate):   # camelCase, no space after comma -- avoid
    return itemPrice*(1+taxRate)
'''
print('Non-PEP8 style (illustrative only):', non_compliant_example.strip().splitlines()[1])


10.8
Non-PEP8 style (illustrative only):     return itemPrice*(1+taxRate)


### Item 3 — Never Expect Python to Detect Errors at Compile Time

Python only checks syntax before running — it does not check that a function exists, that an attribute is spelled correctly, or that types match, until the exact line executes. A typo in a rarely-used branch can sit undetected for a long time. This is why runtime testing (and tools like `mypy` for static type checking, covered later in Item 124) matter more in Python than in a compiled language.


In [3]:
def risky_function(flag):
    if flag:
        return 1 + 1
    else:
        return 1 + undefined_variable   # a real bug -- but Python won't complain until this line runs

print(risky_function(True))    # this path runs fine -- the bug is never triggered
try:
    risky_function(False)       # only NOW does Python discover the problem
except NameError as e:
    print('Discovered only at runtime:', e)


2
Discovered only at runtime: name 'undefined_variable' is not defined


### Item 4 — Write Helper Functions Instead of Complex Expressions

Python lets you chain boolean logic, comprehensions, and conditional expressions into a single dense line — but dense isn't the same as clear. The moment an expression needs a comment to explain what it does, it should be a named helper function instead: it's testable on its own, reusable, and its name documents the intent.


In [4]:
from urllib.parse import parse_qs

query = 'red=5&blue=0&green='
params = parse_qs(query, keep_blank_values=True)

# Dense and hard to read at a glance:
red_dense = int(params['red'][0]) if 'red' in params else 0

# Same logic, as a helper function -- the name documents intent, and it's testable alone
def get_first_int(values, key, default=0):
    found = values.get(key, [''])
    if found[0]:
        return int(found[0])
    return default

print(red_dense)
print(get_first_int(params, 'red'))
print(get_first_int(params, 'green'))     # blank value -> default
print(get_first_int(params, 'opacity'))   # missing key -> default


5
5
0
0


### Item 5 — Prefer Multiple-Assignment Unpacking over Indexing

Instead of pulling items out of a tuple/list one index at a time (`item[0]`, `item[1]`, ...), unpack them all in one assignment: `first, second = pair`. It's shorter, and — critically — it fails loudly with a clear `ValueError` if the number of values doesn't match what you expected, instead of silently returning wrong data via a bad index.


In [5]:
snack = ('bacon', 350)

# Indexing: works, but says nothing about what each position means
name_indexed = snack[0]
calories_indexed = snack[1]
print(name_indexed, calories_indexed)

# Unpacking: same result, self-documenting, and validates the count
name, calories = snack
print(name, calories)

try:
    a, b, c = snack   # snack only has 2 items -- unpacking catches the mismatch immediately
except ValueError as e:
    print('Unpacking caught the mismatch:', e)


bacon 350
bacon 350
Unpacking caught the mismatch: not enough values to unpack (expected 3, got 2)


### Item 6 — Always Surround Single-Element Tuples with Parentheses

A tuple is defined by its **comma**, not its parentheses — `(42)` is just the integer 42, but `(42,)` is a one-element tuple. This is easy to get wrong, especially when a trailing comma is added or removed during editing. Always writing the parentheses even though they're technically optional makes the intent to create a tuple visually unambiguous.


In [6]:
not_a_tuple = (42)
single_tuple = (42,)         # the comma is what matters; the parens just make it visually clear

print(type(not_a_tuple), not_a_tuple)
print(type(single_tuple), single_tuple)

# The same trap without parens at all -- still valid, but easy to misread
also_a_tuple = 42,
print(type(also_a_tuple))


<class 'int'> 42
<class 'tuple'> (42,)
<class 'tuple'>


### Item 7 — Consider Conditional Expressions for Simple Inline Logic

Python's ternary/conditional expression `value_if_true if condition else value_if_false` replaces a 4-line `if`/`else` block that only assigns a variable — but only when both branches are simple. If either branch needs more than a short expression, a real `if` statement is clearer; nesting conditional expressions is a readability trap to avoid.


In [7]:
age = 15

# Full if/else, just to assign one variable:
if age >= 18:
    category = 'adult'
else:
    category = 'minor'
print(category)

# Same thing, as a conditional expression -- fine because both branches are simple
category = 'adult' if age >= 18 else 'minor'
print(category)

# Nesting them gets unreadable fast -- avoid this style:
score = 72
grade = 'A' if score >= 90 else ('B' if score >= 80 else ('C' if score >= 70 else 'F'))
print(grade)   # works, but a normal if/elif chain would read far better here


minor
minor
C


### Item 8 — Prevent Repetition with Assignment Expressions

The walrus operator `:=` (Python 3.8+) assigns a variable *and* produces its value in the same expression — most useful inside an `if` or `while` condition, where you'd otherwise have to compute something, store it, then check it as two separate steps (or worse, compute it twice).


In [8]:
fruits = {'apple': 3, 'banana': 0, 'cherry': 5}

# Without walrus: compute, store, then check -- two steps, or repeat the lookup
count = fruits.get('banana', 0)
if count:
    print(f'{count} banana(s) in stock')
else:
    print('Out of stock, ordering more')

# With walrus: assign AND test in the same condition
if (count := fruits.get('apple', 0)):
    print(f'{count} apple(s) in stock')

# Classic use: avoiding a duplicate call inside a while loop
import random
random.seed(1)
values = iter([5, 3, 8, 0])
def next_value():
    return next(values, 0)

while (v := next_value()) != 0:
    print('got value:', v)


Out of stock, ordering more
3 apple(s) in stock
got value: 5
got value: 3
got value: 8


### Item 9 — Consider `match` for Destructuring in Flow Control; Avoid When `if` Statements Are Sufficient

Python's `match`/`case` (3.10+) shines when you need to check a value's **shape** and pull pieces out of it at the same time — matching a tuple's length, a dict's keys, or a class's type, with automatic destructuring in each `case`. For a simple value-equality check, a plain `if`/`elif` chain (or even a dict lookup) is more obviously readable — `match` isn't a universal replacement for `if`, just a better tool for structural matching.


In [9]:
def describe_point(point):
    match point:
        case (0, 0):
            return 'origin'
        case (x, 0):
            return f'on the x-axis at {x}'
        case (0, y):
            return f'on the y-axis at {y}'
        case (x, y):
            return f'at ({x}, {y})'
        case _:
            return 'not a 2D point'

print(describe_point((0, 0)))
print(describe_point((5, 0)))
print(describe_point((0, -3)))
print(describe_point((2, 4)))

# Simple value equality: a plain if/elif is clearer than forcing a match here
def status_label(code):
    if code == 200:
        return 'OK'
    elif code == 404:
        return 'Not Found'
    else:
        return 'Unknown'

print(status_label(404))


origin
on the x-axis at 5
on the y-axis at -3
at (2, 4)
Not Found
